In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

if os.path.exists('/workspace/data'):
    DATA_DIR      = Path('/workspace/data')
    WORKSPACE_DIR = Path('/workspace')
elif os.path.exists('../environment/data'):
    DATA_DIR      = Path('../environment/data')
    WORKSPACE_DIR = Path('..')
elif os.path.exists('environment/data'):
    DATA_DIR      = Path('environment/data')
    WORKSPACE_DIR = Path('.')
else:
    DATA_DIR      = Path('data')
    WORKSPACE_DIR = Path('.')

WEEKLY_OT_THRESHOLD = 40.0

In [ ]:
shifts    = pd.read_csv(DATA_DIR / 'shifts.csv',       parse_dates=['shift_start', 'shift_end'])
employees = pd.read_csv(DATA_DIR / 'employees.csv')
depts     = pd.read_csv(DATA_DIR / 'departments.csv')
budget    = pd.read_csv(DATA_DIR / 'weekly_budget.csv')

In [ ]:
def week_start_of(ts_series):
    """Return the Monday midnight that begins the ISO week for each timestamp."""
    return (ts_series - pd.to_timedelta(ts_series.dt.weekday, unit='D')).dt.normalize()

# Count cross-week shifts from the raw file (required notebook variable)
start_wk = week_start_of(shifts['shift_start'])
end_wk   = week_start_of(shifts['shift_end'])
cross_week_shift_count = int((start_wk != end_wk).sum())

In [ ]:
# Split shifts that span a week boundary at Monday 00:00.
# Hours before the boundary count toward the ending week;
# hours from the boundary onward count toward the starting week.
crosses     = start_wk != end_wk
normal      = shifts[~crosses].copy()
normal['seg_start'] = normal['shift_start']
normal['seg_end']   = normal['shift_end']

xw = shifts[crosses].copy()
xw['boundary'] = week_start_of(xw['shift_end'])   # Monday midnight that falls inside the shift

seg1 = xw.copy()
seg1['seg_start'] = xw['shift_start']
seg1['seg_end']   = xw['boundary']

seg2 = xw.copy()
seg2['seg_start'] = xw['boundary']
seg2['seg_end']   = xw['shift_end']

segments = pd.concat([normal, seg1, seg2], ignore_index=True)
segments['hours']      = (segments['seg_end'] - segments['seg_start']).dt.total_seconds() / 3600
segments['week_start'] = week_start_of(segments['seg_start']).dt.strftime('%Y-%m-%d')

In [ ]:
# Total hours per employee per week
weekly_hrs = (
    segments
    .groupby(['employee_id', 'week_start'])['hours']
    .sum()
    .reset_index()
    .rename(columns={'hours': 'total_hours'})
)

In [ ]:
# Apply weekly overtime threshold
weekly_hrs['regular_hours'] = np.minimum(weekly_hrs['total_hours'], WEEKLY_OT_THRESHOLD)
weekly_hrs['ot_hours']      = np.maximum(weekly_hrs['total_hours'] - WEEKLY_OT_THRESHOLD, 0.0)

In [ ]:
# Merge employee rate and compute individual weekly cost
weekly_hrs = weekly_hrs.merge(employees[['employee_id', 'department_id', 'hourly_rate']],
                               on='employee_id')
weekly_hrs['actual_cost'] = (
    weekly_hrs['regular_hours'] * weekly_hrs['hourly_rate']
    + weekly_hrs['ot_hours']    * weekly_hrs['hourly_rate'] * 1.5
)

In [ ]:
# Aggregate to department + week level
dept_week = (
    weekly_hrs
    .groupby(['department_id', 'week_start'])
    .agg(actual_cost=('actual_cost', 'sum'),
         ot_hours=('ot_hours', 'sum'))
    .reset_index()
)

In [ ]:
# Build variance report
budget['budgeted_cost'] = budget['budgeted_hours'] * budget['avg_hourly_rate']
report = budget.merge(dept_week, on=['department_id', 'week_start'])
report = report.merge(depts, on='department_id')
report['variance'] = report['actual_cost'] - report['budgeted_cost']

report = report[[
    'department_id', 'department_name', 'week_start',
    'budgeted_cost', 'actual_cost', 'variance'
]].sort_values(['department_id', 'week_start']).reset_index(drop=True)

In [ ]:
report.to_csv(WORKSPACE_DIR / 'payroll_variance_report.csv', index=False)

In [ ]:
total_budgeted_cost    = float(round(report['budgeted_cost'].sum(), 2))
total_actual_cost      = float(round(report['actual_cost'].sum(), 2))
total_variance         = float(round(total_actual_cost - total_budgeted_cost, 2))
total_overtime_hours   = float(round(dept_week['ot_hours'].sum(), 2))
over_budget_week_count = int((report['variance'] > 0).sum())